# الذكاء الاصطناعي في الهندسة المدنية — الفصل 6 Deep learning (a)
Companion notebook. Run all cells (Runtime → Run all). Every number in the book comes from this code.

In [ ]:
!pip -q install xgboost shap scikit-image statsmodels ucimlrepo
import os; os.makedirs('fig', exist_ok=True)

In [ ]:
%%writefile crackgen.py
import numpy as np
from scipy.ndimage import gaussian_filter
def concrete_bg(rng, n=64):
    img = 0.55 + 0.08*gaussian_filter(rng.normal(0,1,(n,n)), 1.2) + 0.03*rng.normal(0,1,(n,n))
    for _ in range(rng.integers(3,12)):                      # pores
        y,x = rng.integers(0,n,2); r=rng.uniform(0.6,1.6)
        yy,xx=np.ogrid[:n,:n]; img[(yy-y)**2+(xx-x)**2<=r*r] -= rng.uniform(0.1,0.25)
    gy,gx = np.mgrid[:n,:n]/n; img += rng.uniform(-0.12,0.12)*gx + rng.uniform(-0.12,0.12)*gy  # lighting
    if rng.random()<0.3:                                      # shadow band
        a=rng.uniform(0,np.pi); d=(gx-0.5)*np.cos(a)+(gy-0.5)*np.sin(a); img[d>rng.uniform(-0.3,0.3)] -= rng.uniform(0.08,0.18)
    return img
def draw_path(img, pts, width, depth):
    n=img.shape[0]; yy,xx=np.mgrid[:n,:n]; m=np.zeros_like(img)
    for (y0,x0),(y1,x1) in zip(pts[:-1],pts[1:]):
        for t in np.linspace(0,1,8):
            y=y0+(y1-y0)*t; x=x0+(x1-x0)*t
            m=np.maximum(m, np.exp(-((yy-y)**2+(xx-x)**2)/(2*(width/2)**2)))
    return img - depth*m, m>0.5
def crack(img, rng):
    n=img.shape[0]; y,x=rng.uniform(16,48,2); a=rng.uniform(0,2*np.pi); pts=[(y,x)]
    for _ in range(rng.integers(8,16)):
        a+=rng.normal(0,0.6); y+=3.5*np.sin(a); x+=3.5*np.cos(a)
        if not (4<y<60 and 4<x<60): a+=np.pi; y=np.clip(y,4,60); x=np.clip(x,4,60)
        pts.append((y,x))
    return draw_path(img, pts, rng.uniform(0.8,2.6), rng.uniform(0.15,0.45))
def joint(img, rng):                                           # straight formwork joint = hard negative
    n=img.shape[0]; a=rng.uniform(0,np.pi); c=rng.uniform(16,48)
    p0=(c-40*np.sin(a), c-40*np.cos(a)); p1=(c+40*np.sin(a), c+40*np.cos(a))
    return draw_path(img,[p0,p1],rng.uniform(1.0,2.2),rng.uniform(0.15,0.4))
def make_images(N, seed=0, joints=True):
    rng=np.random.default_rng(seed); X=np.zeros((N,64,64),np.float32); y=np.zeros(N,np.int64); M=np.zeros((N,64,64),bool)
    for i in range(N):
        img=concrete_bg(rng)
        if rng.random()<0.5: img,M[i]=crack(img,rng); y[i]=1
        elif joints and rng.random()<0.4: img,_=joint(img,rng)
        X[i]=np.clip(img,0,1)
    return X,y,M


In [ ]:
import numpy as np, torch, torch.nn as nn, time, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from crackgen import make_images
torch.manual_seed(0); torch.set_num_threads(2)
Xtr,ytr,_=make_images(4000,1); Xte,yte,_=make_images(1000,2)
norm=lambda X:(X-X.mean(axis=(1,2),keepdims=True))/(X.std(axis=(1,2),keepdims=True)+1e-6)
print("train cracks:",ytr.mean().round(3))
fig,ax=plt.subplots(3,6,figsize=(8,4.3)); rng=np.random.default_rng(9)
idx=np.r_[np.where(ytr==1)[0][:6],np.where(ytr==0)[0][:12]]
for a,i in zip(ax.ravel(),idx): a.imshow(Xtr[i],cmap="gray",vmin=0,vmax=1); a.set_axis_off(); a.set_title("crack" if ytr[i] else "no crack",fontsize=7)
fig.tight_layout(); fig.savefig("fig/fig6_1_samples.png",dpi=200)
class CNN(nn.Module):
    def __init__(s):
        super().__init__()
        s.f=nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.BatchNorm2d(16),nn.ReLU(),nn.MaxPool2d(2),
                          nn.Conv2d(16,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
                          nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.AdaptiveMaxPool2d(1))
        s.h=nn.Linear(64,2)
    def forward(s,x): return s.h(s.f(x).flatten(1))
m=CNN(); print("parameters:",sum(p.numel() for p in m.parameters()))
opt=torch.optim.Adam(m.parameters(),2e-3); lf=nn.CrossEntropyLoss()
Xt=torch.tensor(norm(Xtr))[:,None]; Yt=torch.tensor(ytr); Xv=torch.tensor(norm(Xte))[:,None]; Yv=torch.tensor(yte)
hist=[]; t0=time.time()
for ep in range(12):
    m.train(); perm=torch.randperm(len(Xt)); L=0
    for b in range(0,len(Xt),64):
        i=perm[b:b+64]; opt.zero_grad(); l=lf(m(Xt[i]),Yt[i]); l.backward(); opt.step(); L+=l.item()*len(i)
    m.eval()
    with torch.no_grad(): pv=m(Xv).argmax(1); vl=lf(m(Xv),Yv).item()
    acc=(pv==Yv).float().mean().item(); hist.append([L/len(Xt),vl,acc])
    print(f"epoch {ep+1:2d} train_loss={L/len(Xt):.3f} test_loss={vl:.3f} test_acc={acc:.3f}")
print("time s:",round(time.time()-t0))
p=pv.numpy(); tp=((p==1)&(yte==1)).sum(); fn=((p==0)&(yte==1)).sum(); fp=((p==1)&(yte==0)).sum(); tn=((p==0)&(yte==0)).sum()
print("confusion TP FN FP TN:",tp,fn,fp,tn, " recall",round(tp/(tp+fn),3)," precision",round(tp/(tp+fp),3))
H=np.array(hist); fig,ax=plt.subplots(1,2,figsize=(8,3))
ax[0].plot(range(1,13),H[:,0],label="train",color="#2b6cb0"); ax[0].plot(range(1,13),H[:,1],label="test",color="#dd6b20"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].legend(frameon=False)
ax[1].plot(range(1,13),H[:,2],color="#2b6cb0"); ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Test accuracy")
for a in ax: a.spines[["top","right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig/fig6_2_curves.png",dpi=200)
torch.save(m.state_dict(),"cnn_crack.pt")
# domain shift: test on images without the hard-negative joints vs with only joints
Xj,yj,_=make_images(1000,3,joints=True)
with torch.no_grad(): pj=m(torch.tensor(norm(Xj))[:,None]).argmax(1).numpy()